In [ ]:
import pandas as pd
import re
import json
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import nltk
from tabulate import tabulate
from rich.console import Console
from rich.table import Table

nltk.download("punkt")
console = Console()

# ==================================================
# LOAD DATA JSONL LIPUTAN6
# ==================================================
jsonl_file = "liputan6.jsonl"
articles = []

with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            articles.append(json.loads(line))
        except json.JSONDecodeError:
            continue

print(f"✅ Total artikel loaded: {len(articles)}")
 

# ==================================================
# FUNGSI BANTU
# ==================================================
def split_paragraphs(text):
    if not text:
        return []
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


# ==================================================
# PROSES DATASET ENTAILMENT (ONLY WITH SUMMARY)
# ==================================================
output_jsonl = "liputan6_entailment_dataset.jsonl"
output_csv = "liputan6_entailment_dataset.csv"

records_count = 0
premise_count = 0
hypothesis_sentence_count = 0
skipped = 0

csv_rows = []

with open(output_jsonl, "w", encoding="utf-8") as out_f:
    for idx, art in enumerate(tqdm(articles, desc="Memproses artikel")):

        summary = art.get("summary", "")
        content = art.get("content", "")

        # ❌ SKIP jika tidak ada summary
        if not summary or summary.strip().lower() in ["", "n/a", "none"]:
            skipped += 1
            continue

        doc_id = f"liputan6_{idx+1:05d}"
        title = art.get("title", "")

        premise_paragraphs = split_paragraphs(summary)
        hypothesis_paragraphs = split_paragraphs(content)

        premise_count += len(premise_paragraphs)

        for p_id, premise in enumerate(premise_paragraphs, start=1):
            for h_id, hyp_paragraph in enumerate(hypothesis_paragraphs, start=1):

                hyp_sentences = sent_tokenize(hyp_paragraph)
                hypothesis_sentence_count += len(hyp_sentences)

                for hs_id, hypothesis in enumerate(hyp_sentences, start=1):

                    record = {
                        "doc_id": doc_id,
                        "title": title,
                        "paragraph_premise": p_id,
                        "paragraph_hypothesis": h_id,
                        "sentence_hypothesis": hs_id,
                        "premise": premise,
                        "hypothesis": hypothesis,
                        "content": content
                    }

                    # JSONL (streaming)
                    json.dump(record, out_f, ensure_ascii=False)
                    out_f.write("\n")

                    # CSV buffer
                    csv_rows.append(record)

                    records_count += 1


# ==================================================
# SAVE CSV
# ==================================================
df = pd.DataFrame(csv_rows)
df.to_csv(output_csv, index=False, encoding="utf-8")

print(f"\n✅ JSONL saved: {output_jsonl}")
print(f"✅ CSV saved: {output_csv}")

print(f"\n📊 Artikel dipakai: {len(articles) - skipped}")
print(f"❌ Artikel dibuang (tanpa summary): {skipped}")


# ==================================================
# STATISTIK
# ==================================================
stats_table = Table(title="Statistik Dataset Entailment Liputan6")

stats_table.add_column("Metrik")
stats_table.add_column("Nilai", justify="right")

stats_table.add_row("Total Artikel Awal", str(len(articles)))
stats_table.add_row("Artikel Dipakai", str(len(articles) - skipped))
stats_table.add_row("Artikel Dibuang", str(skipped))
stats_table.add_row("Total Premise", str(premise_count))
stats_table.add_row("Total Kalimat Hypothesis", str(hypothesis_sentence_count))
stats_table.add_row("Total Pair", str(records_count))

if (len(articles) - skipped) > 0:
    stats_table.add_row(
        "Rata-rata pair/artikel",
        f"{records_count / (len(articles) - skipped):.2f}"
    )

console.print(stats_table)


# ==================================================
# PREVIEW (AMAN MEMORY)
# ==================================================
from itertools import islice

preview_rows = []

with open(output_jsonl, "r", encoding="utf-8") as f:
    for line in islice(f, 10):
        preview_rows.append(json.loads(line))

df_preview = pd.DataFrame(preview_rows)

preview_cols = [
    "doc_id",
    "title",
    "paragraph_premise",
    "paragraph_hypothesis",
    "sentence_hypothesis",
    "premise",
    "hypothesis",
]

print("\nContoh 10 baris pertama dataset:\n")
print(
    tabulate(
        df_preview[preview_cols],
        headers="keys",
        tablefmt="fancy_grid",
        showindex=False,
        maxcolwidths=[12, 20, 8, 8, 8, 40, 40]
    )
)1

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\andik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


✅ Total artikel loaded: 6309


Memproses artikel: 100%|██████████| 6309/6309 [00:08<00:00, 707.14it/s] 



✅ JSONL saved: liputan6_entailment_dataset.jsonl
✅ CSV saved: liputan6_entailment_dataset.csv

📊 Artikel dipakai: 5702
❌ Artikel dibuang (tanpa summary): 607


Statistik Dataset Entailment Liputan6
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metrik                   ┃  Nilai ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Total Artikel Awal       │   6309 │
│ Artikel Dipakai          │   5702 │
│ Artikel Dibuang          │    607 │
│ Total Premise            │   5703 │
│ Total Kalimat Hypothesis │ 143274 │
│ Total Pair               │ 143274 │
│ Rata-rata pair/artikel   │  25.13 │
└──────────────────────────┴────────┘


Contoh 10 baris pertama dataset:

╒══════════════╤════════════════════╤═════════════════════╤════════════════════════╤═══════════════════════╤══════════════════════════════════════════╤══════════════════════════════════════════╕
│ doc_id       │ title              │   paragraph_premise │   paragraph_hypothesis │   sentence_hypothesis │ premise                                  │ hypothesis                               │
╞══════════════╪════════════════════╪═════════════════════╪════════════════════════╪═══════════════════════╪══════════════════════════════════════════╪══════════════════════════════════════════╡
│ liputan6_000 │ Kisah Wisudawan    │                   1 │                      1 │                     1 │ Sekolah Lansia DKI Jakarta wadah lansia  │ Liputan6.com, JakartaProgram Sekolah     │
│ 02           │ Sekolah Lansia:    │                     │                        │                       │ belajar, berinteraksi, dan tingkatkan    │ Lansia yang digagas Pemerintah Pr